# Retro Sprite Family Classifier
### Final training and evaluation notebook

**Goal:** classify 24×24 RGB sprite images into 11 selected sprite families using TensorFlow/Keras.

This notebook contains the finalized, reproducible pipeline:
1. Install/import dependencies.
2. Load and prepare the dataset.
3. Train the selected wider CNN.
4. Review validation performance.
5. Evaluate the selected checkpoint on the test set once.
6. Save the model and class labels for deployment in Streamlit.

**Important:** The model was selected after repeated validation experiments. The test set is reserved for the final evaluation and should not be used to choose architectures or tune hyperparameters.

**Reported development result:** best validation accuracy observed in the previous training session was **57.69%**. Training from scratch can produce slightly different results depending on the runtime, TensorFlow version, and hardware.

## 1. Install dependencies

Run this cell in a fresh notebook environment if needed. On Google Colab, TensorFlow is often preinstalled, but the command ensures the main packages are available.

In [ ]:
%pip -q install datasets pillow scikit-learn matplotlib seaborn

## 2. Imports and reproducibility

We set random seeds to improve reproducibility. GPU kernels may still introduce small nondeterministic differences.

In [ ]:
import io
import json
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from datasets import load_dataset
from PIL import Image
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 3. Load the dataset

The source dataset contains sprite images and metadata. We use its predefined `train`, `validation`, and `test` splits.

In [ ]:
DATASET_NAME = "unstonio/pixelgpt-24x24-20k"

dataset = load_dataset(DATASET_NAME)
required_splits = ["train", "validation", "test"]
missing_splits = [split for split in required_splits if split not in dataset]
if missing_splits:
    raise ValueError(f"Dataset is missing required splits: {missing_splits}")

frames = {
    split: dataset[split].to_pandas()
    for split in required_splits
}

for split, frame in frames.items():
    print(f"{split}: {len(frame):,} rows")
print("\nAvailable columns:", list(frames["train"].columns))
print("Number of source families:", frames["train"]["family"].nunique())

## 4. Select the 11 finalized families

To focus the task on a smaller set of sprite families, the project retains the following 11 labels. The order below defines the numeric class IDs used by the model and Streamlit app.

In [ ]:
RETAINED_FAMILIES = [
    "01_animals",
    "02_fantasy_creatures",
    "04_plants_and_fungi",
    "05_food_and_drink",
    "06_containers_and_storage",
    "09_tools_and_crafting",
    "11_machines_and_technology",
    "14_places_and_structures",
    "16_nature_and_landscapes",
    "18_effects_and_celestial",
    "19_symbols_and_documents",
]

NUM_CLASSES = len(RETAINED_FAMILIES)
family_to_id = {family: index for index, family in enumerate(RETAINED_FAMILIES)}
id_to_family = {index: family for family, index in family_to_id.items()}

unknown_families = set(RETAINED_FAMILIES) - set(frames["train"]["family"].unique())
if unknown_families:
    raise ValueError(f"Expected families not found in training data: {sorted(unknown_families)}")

print("Number of classes:", NUM_CLASSES)
for class_id, family in id_to_family.items():
    print(f"{class_id:2d} -> {family}")

## 5. Decode images and prepare train/validation/test arrays

Images are decoded from the dataset's stored image bytes, converted to RGB, and normalized to the range 0–1. We apply the same family filtering and label mapping to every split.

In [ ]:
IMAGE_HEIGHT = 24
IMAGE_WIDTH = 24
CHANNELS = 3

def prepare_split(dataframe):
    images = []
    labels = []

    selected_rows = dataframe[dataframe["family"].isin(RETAINED_FAMILIES)]

    for _, row in selected_rows.iterrows():
        image_value = row["image"]

        # Hugging Face image columns are commonly dictionaries with a "bytes" key.
        if isinstance(image_value, dict):
            image_bytes = image_value.get("bytes")
            if image_bytes is None and image_value.get("path"):
                with open(image_value["path"], "rb") as image_file:
                    image_bytes = image_file.read()
            if image_bytes is None:
                raise ValueError("Could not read image bytes from a dataset row.")
            image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        else:
            # Fallback for rows that already contain a PIL image.
            image = image_value.convert("RGB")

        image = image.resize((IMAGE_WIDTH, IMAGE_HEIGHT), Image.Resampling.NEAREST)
        images.append(np.asarray(image, dtype=np.float32) / 255.0)
        labels.append(family_to_id[row["family"]])

    X = np.asarray(images, dtype=np.float32)
    y = np.asarray(labels, dtype=np.int32)

    if len(X) == 0:
        raise ValueError("No images remained after filtering the selected families.")

    return X, y

X_train_11, y_train_11 = prepare_split(frames["train"])
X_val_11, y_val_11 = prepare_split(frames["validation"])
X_test_11, y_test_11 = prepare_split(frames["test"])

print("Training:", X_train_11.shape, y_train_11.shape)
print("Validation:", X_val_11.shape, y_val_11.shape)
print("Test:", X_test_11.shape, y_test_11.shape)
print("Pixel range:", float(X_train_11.min()), "to", float(X_train_11.max()))

# One-hot labels are required for categorical cross-entropy with label smoothing.
y_train_onehot = keras.utils.to_categorical(y_train_11, num_classes=NUM_CLASSES)
y_val_onehot = keras.utils.to_categorical(y_val_11, num_classes=NUM_CLASSES)

## 6. Inspect class distribution

This helps explain whether some families have fewer training examples than others.

In [ ]:
train_counts = pd.Series(y_train_11).value_counts().reindex(range(NUM_CLASSES), fill_value=0)
distribution = pd.DataFrame({
    "class_id": range(NUM_CLASSES),
    "family": RETAINED_FAMILIES,
    "training_images": train_counts.values,
})
display(distribution)

plt.figure(figsize=(12, 5))
sns.barplot(data=distribution, x="family", y="training_images")
plt.title("Training Images per Selected Family")
plt.xlabel("Sprite family")
plt.ylabel("Number of images")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

## 7. Build the finalized wider CNN

The model uses horizontal-flip and small-translation augmentation, convolutional feature extraction, batch normalization, spatial dropout, global average pooling, and a regularized dense layer. Label smoothing is used in the loss function.

In [ ]:
def build_wide_cnn():
    weight_decay = 0.0005

    model = keras.Sequential([
        layers.Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, CHANNELS)),

        # Augmentation is active during training and inactive during inference.
        layers.RandomFlip("horizontal"),
        layers.RandomTranslation(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="constant",
            fill_value=0.0
        ),

        # Convolutional block 1
        layers.Conv2D(
            48, 3, padding="same", use_bias=False,
            kernel_regularizer=regularizers.l2(weight_decay)
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Conv2D(
            48, 3, padding="same", use_bias=False,
            kernel_regularizer=regularizers.l2(weight_decay)
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(),
        layers.SpatialDropout2D(0.10),

        # Convolutional block 2
        layers.Conv2D(
            96, 3, padding="same", use_bias=False,
            kernel_regularizer=regularizers.l2(weight_decay)
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Conv2D(
            96, 3, padding="same", use_bias=False,
            kernel_regularizer=regularizers.l2(weight_decay)
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(),
        layers.SpatialDropout2D(0.15),

        # Convolutional block 3
        layers.Conv2D(
            128, 3, padding="same", use_bias=False,
            kernel_regularizer=regularizers.l2(weight_decay)
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        # Classifier head
        layers.GlobalAveragePooling2D(),
        layers.Dense(
            128,
            activation="relu",
            kernel_regularizer=regularizers.l2(weight_decay)
        ),
        layers.Dropout(0.45),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ], name="wide_cnn_11")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0004),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"],
    )
    return model

model = build_wide_cnn()
model.summary()

## 8. Train and save the best checkpoint

The checkpoint is selected by **highest validation accuracy**, so the exported model corresponds to the best validation-accuracy epoch. Early stopping monitors validation loss to limit overfitting; the checkpoint file preserves the best validation-accuracy model.

In [ ]:
MODEL_PATH = "best_wide_11.keras"

callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    X_train_11,
    y_train_onehot,
    validation_data=(X_val_11, y_val_onehot),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

## 9. Review training and validation curves

In [ ]:
history_df = pd.DataFrame(history.history)
best_val_accuracy = float(history_df["val_accuracy"].max())
best_accuracy_epoch = int(history_df["val_accuracy"].idxmax() + 1)
lowest_val_loss = float(history_df["val_loss"].min())

print(f"Best validation accuracy: {best_val_accuracy * 100:.2f}%")
print(f"Epoch with best validation accuracy: {best_accuracy_epoch}")
print(f"Lowest validation loss: {lowest_val_loss:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_df["accuracy"], label="Training accuracy")
axes[0].plot(history_df["val_accuracy"], label="Validation accuracy")
axes[0].set_title("Accuracy by Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history_df["loss"], label="Training loss")
axes[1].plot(history_df["val_loss"], label="Validation loss")
axes[1].set_title("Loss by Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Final test evaluation

Run this evaluation only after architecture and hyperparameters have been finalized. Do not use the test results to choose another model or tune settings.

In [ ]:
final_model = keras.models.load_model(MODEL_PATH)

test_loss, test_accuracy = final_model.evaluate(X_test_11, y_test_11, verbose=1)
print(f"Final test accuracy: {test_accuracy * 100:.2f}%")
print(f"Final test loss: {test_loss:.4f}")

test_probabilities = final_model.predict(X_test_11, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)

print("\nPer-family classification report:")
print(classification_report(
    y_test_11,
    test_predictions,
    labels=np.arange(NUM_CLASSES),
    target_names=RETAINED_FAMILIES,
    digits=3,
    zero_division=0,
))

test_cm = confusion_matrix(
    y_test_11,
    test_predictions,
    labels=np.arange(NUM_CLASSES),
)

plt.figure(figsize=(12, 9))
sns.heatmap(
    test_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=RETAINED_FAMILIES,
    yticklabels=RETAINED_FAMILIES,
)
plt.xlabel("Predicted family")
plt.ylabel("Actual family")
plt.title("Final Test Confusion Matrix")
plt.xticks(rotation=75, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Export the model and label map for the Streamlit app

The Streamlit app expects `final_sprite_classifier.keras` and `class_names.json` in the same project folder. Download these files from the notebook's runtime and place them beside `app.py`.

In [ ]:
final_model.save("final_sprite_classifier.keras")

with open("class_names.json", "w", encoding="utf-8") as label_file:
    json.dump(RETAINED_FAMILIES, label_file, indent=2)

print("Saved: final_sprite_classifier.keras")
print("Saved: class_names.json")
print("\nExport these two files together with app.py and requirements.txt.")

## 12. Project conclusion

This project demonstrates an end-to-end image-classification workflow for low-resolution pixel-art sprites. It includes data filtering, preprocessing, CNN training, validation monitoring, a held-out test evaluation, and export for a Streamlit demo.

The best validation accuracy observed during development was **57.69%**. This is a useful baseline, but it is not evidence that the model is production-ready. Sprite families can look visually similar, and the reduced 11-class task is narrower than the original 19-family dataset. Report the test accuracy printed in Section 10 as the final held-out result.